# Asistente de soporte técnico de TI: clasificador de intenciones y clasificador visual
MBIA2602 - Proyecto Fase 2

Para ejecutar: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4). Después, Ejecutar todo.

In [ ]:
import io
import os
import random
import time
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

from nltk.stem import SnowballStemmer
from nltk.tokenize import RegexpTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

## Parte A. Componente conversacional

### A1. Conjunto de datos textual

In [ ]:
# Conjunto de datos textual: consultas de ejemplo por intención (soporte técnico de TI)
DATOS = {
    "saludo": [
        "hola", "buenos días", "buenas tardes", "buenas noches", "qué tal",
        "hola, buen día", "saludos", "hey, hola", "hola buenas", "buen día equipo",
        "qué onda", "hola, ¿hay alguien ahí?", "hola, necesito un favor",
        "buenas, ¿me pueden atender?", "hola asistente", "holaa", "muy buenos días",
        "hola de nuevo", "hola, ¿cómo estás?", "buen día, una consulta",
        "saludos cordiales", "hola, buenas tardes a todos", "ey, hola", "hola, ¿qué tal?",
    ],
    "ayuda": [
        "ayuda", "¿qué puedes hacer?", "¿en qué me puedes ayudar?", "muéstrame las opciones",
        "menú", "¿con qué temas me ayudas?", "no sé qué preguntar", "¿qué servicios ofreces?",
        "necesito ayuda", "¿cómo funciona este asistente?", "¿qué problemas resuelves?",
        "dime qué opciones hay", "lista de temas por favor", "¿qué consultas atiendes?",
        "ayúdame", "¿para qué sirves?", "quiero ver qué puedes resolver",
        "¿cuáles son tus funciones?", "¿qué me puedes ofrecer?", "ayuda por favor",
        "¿qué opciones tengo?", "¿puedes orientarme?", "explícame cómo usarte",
        "¿sobre qué puedo preguntarte?",
    ],
    "contrasena": [
        "olvidé mi contraseña", "no recuerdo mi contraseña", "mi cuenta está bloqueada",
        "necesito restablecer mi clave", "cómo cambio mi contraseña", "se me bloqueó el usuario",
        "perdí mi password", "quiero resetear mi contraseña", "mi contraseña expiró",
        "no puedo iniciar sesión porque olvidé la clave", "desbloquear mi cuenta",
        "la contraseña no es válida", "me pide cambiar la contraseña y no puedo",
        "cuenta bloqueada por intentos fallidos", "cómo recupero mi acceso",
        "mi usuario quedó bloqueado", "contraseña incorrecta aunque es la correcta",
        "restablecer password del equipo", "necesito una nueva contraseña",
        "olvidé mis credenciales", "se venció mi contraseña", "no me deja entrar con mi clave",
        "cambiar clave de acceso", "mi cuenta de red está bloqueada",
    ],
    "vpn": [
        "no puedo conectarme a la vpn", "la vpn no conecta", "error al abrir el cliente vpn",
        "necesito acceso remoto a la empresa", "se desconecta la vpn", "la vpn marca error 800",
        "cómo me conecto desde casa a la red de la oficina", "problemas con el túnel vpn",
        "mi vpn está muy lenta", "la vpn dice credenciales incorrectas",
        "no abre el cliente de vpn", "trabajo remoto y no tengo acceso",
        "se cae la conexión vpn a cada rato", "configurar vpn en mi laptop",
        "no logro entrar a los sistemas desde casa", "la vpn se queda en conectando",
        "vpn no funciona", "acceso remoto fallando", "instalar el cliente de vpn",
        "la vpn me da timeout", "no carga la vpn",
        "teletrabajo sin conexión a la red corporativa", "error de autenticación en la vpn",
        "no puedo trabajar en remoto",
    ],
    "wifi": [
        "no tengo internet", "el wifi no funciona", "no hay señal de wifi",
        "mi conexión a internet está muy lenta", "no puedo conectarme a la red",
        "el internet se cae seguido", "la red inalámbrica no aparece", "se corta el wifi",
        "estoy sin conexión", "el router no tiene luz", "problemas con la red de la oficina",
        "no me conecta al wifi", "el internet va muy lento", "mi computadora no detecta redes",
        "wifi conectado pero sin internet", "las páginas no cargan",
        "no hay conexión en mi escritorio", "el cable de red no funciona",
        "falla la conexión a internet", "no encuentro la red del trabajo",
        "la señal del wifi es muy débil", "internet intermitente", "se me va el wifi",
        "no puedo navegar",
    ],
    "impresora": [
        "mi impresora no imprime", "la impresora no responde", "se atoró el papel",
        "no hay tóner", "necesito imprimir un documento y no puedo",
        "la impresora imprime en blanco", "no encuentro la impresora en mi equipo",
        "el escáner no funciona", "la impresora sale con rayas",
        "se quedó trabada la cola de impresión", "falla la impresora de mi piso",
        "no conecta la impresora", "cómo agrego una impresora", "imprime muy lento",
        "la impresora marca error", "se acabó la tinta", "no puedo escanear documentos",
        "la impresión sale cortada", "el trabajo de impresión queda en espera",
        "impresora desconectada", "la copiadora no funciona",
        "instalar el controlador de la impresora", "imprime hojas en negro",
        "la impresora hace ruido y no jala",
    ],
    "software": [
        "necesito instalar un programa", "quiero instalar office", "cómo solicito una licencia",
        "necesito el software de diseño", "instalar una aplicación en mi laptop",
        "me pueden instalar python", "solicito acceso a un programa",
        "necesito actualizar mi software", "no me deja instalar", "licencia de adobe",
        "quiero que me instalen zoom", "cómo pido un programa nuevo",
        "se venció la licencia de mi programa", "falta un programa en mi equipo",
        "necesito autorización para instalar software", "instalar teams",
        "pedir una herramienta de trabajo", "actualización de windows pendiente",
        "requiero una aplicación para mi área", "puedo instalar mi propio programa",
        "el programa pide una licencia", "instalación de antivirus",
        "necesito un programa para editar pdf", "desinstalar una aplicación",
    ],
    "correo": [
        "outlook no abre", "no me llegan los correos", "mi buzón está lleno",
        "no puedo enviar correos", "el correo se queda en la bandeja de salida",
        "outlook se cierra solo", "no sincroniza mi correo", "cómo configuro mi correo",
        "error al adjuntar un archivo", "no recibo mensajes", "mi correo no carga",
        "se perdieron mis correos", "outlook pide contraseña todo el tiempo",
        "mi calendario no se actualiza", "no puedo abrir un adjunto",
        "los correos tardan en llegar", "problema con la firma del correo", "el correo rebota",
        "buzón compartido no aparece", "outlook está en modo sin conexión",
        "no puedo acceder a mi email", "correo corporativo no funciona",
        "se llenó mi bandeja", "configurar correo en el celular",
    ],
    "equipo_lento": [
        "mi computadora está muy lenta", "la laptop se congela",
        "el equipo tarda mucho en encender", "se traba todo", "mi pc va lentísima",
        "se me cierra el programa solo", "la computadora se reinicia sola",
        "tarda mucho en abrir los archivos", "mi laptop se calienta demasiado",
        "se queda colgada la computadora", "el ventilador hace mucho ruido",
        "no tengo espacio en el disco", "el sistema responde con retraso",
        "pantalla azul en mi equipo", "mi equipo se apaga de repente",
        "tarda una eternidad en iniciar", "la laptop va muy despacio", "windows se pone lento",
        "la computadora no responde", "excel tarda mucho en abrir",
        "mi máquina se traba al abrir varios programas",
        "el equipo está lento desde la actualización", "mi laptop tarda en responder",
        "la computadora está pesada",
    ],
    "pantalla": [
        "mi monitor no enciende", "la pantalla se ve negra", "no hay imagen en el monitor",
        "la pantalla parpadea", "el monitor dice sin señal", "se ve borrosa la pantalla",
        "tengo rayas en la pantalla", "mi segundo monitor no se detecta",
        "la pantalla está muy oscura", "el monitor se apaga solo",
        "no se ve nada en la pantalla", "cómo conecto un monitor externo",
        "la resolución se ve mal", "la pantalla se ve con colores raros",
        "el cable hdmi no da imagen", "pantalla rota", "el brillo no cambia",
        "mi pantalla tiene manchas", "el monitor no funciona", "la pantalla tiembla",
        "no reconoce el monitor", "aparece sin señal en mi pantalla",
        "necesito un monitor nuevo", "la imagen se ve estirada",
    ],
    "horario": [
        "¿cuál es el horario de soporte?", "¿a qué hora atienden?",
        "¿hasta qué hora hay soporte?", "¿trabajan los sábados?", "¿están abiertos hoy?",
        "¿cuándo puedo llamar a soporte?", "horario de atención",
        "¿atienden en fin de semana?", "¿a qué hora abren?", "¿a qué hora cierran?",
        "¿hay soporte por la noche?", "¿qué días dan servicio?",
        "¿hay alguien de soporte en la tarde?", "dime el horario",
        "¿en qué horario puedo reportar un problema?", "¿atienden los domingos?",
        "¿cuál es su horario de servicio?", "¿hasta cuándo puedo pedir ayuda hoy?",
        "¿están disponibles ahora?", "¿cuál es el horario de la mesa de ayuda?",
        "¿a qué hora empieza el soporte?", "¿trabajan en días festivos?",
        "¿cuándo me pueden atender?", "horarios de la mesa de servicio",
    ],
    "ticket": [
        "quiero abrir un ticket", "levanta un reporte por favor",
        "necesito hablar con una persona", "quiero que me atienda un agente",
        "genera un incidente", "crear un ticket de soporte", "pásame con un técnico",
        "quiero reportar una falla", "necesito que alguien venga a mi lugar", "abre un caso",
        "no se resolvió, quiero un ticket", "levantar un reporte de mi problema",
        "quiero hablar con soporte", "registrar un incidente", "necesito atención humana",
        "comunícame con una persona real",
        "quiero dar seguimiento a mi problema con un ticket", "mándame un técnico",
        "solicito un ticket", "ya intenté todo, necesito un agente", "reportar un problema",
        "genera un folio", "quiero levantar un caso",
        "escala mi problema con un especialista",
    ],
    "agradecimiento": [
        "gracias", "muchas gracias", "te lo agradezco", "perfecto, gracias",
        "excelente, muchas gracias", "eso era todo, gracias", "mil gracias",
        "gracias por la ayuda", "muy amable, gracias", "listo, gracias", "ok gracias",
        "genial, gracias", "gracias, me sirvió", "súper, gracias", "agradezco tu ayuda",
        "gracias por tu tiempo", "qué bien, gracias", "gracias, ya quedó",
        "te agradezco mucho", "muchas gracias por la información",
        "excelente servicio, gracias", "gracias, resolvió mi duda", "buenísimo, gracias",
        "gracias, eso me ayuda",
    ],
    "despedida": [
        "adiós", "hasta luego", "nos vemos", "bye", "chao", "hasta pronto", "me despido",
        "que tengas buen día", "hasta mañana", "ya me voy", "eso es todo, adiós", "salir",
        "terminar conversación", "cerrar el chat", "nos vemos luego", "adiós, gracias",
        "hasta la próxima", "bye bye", "me retiro", "que estés bien",
        "ya no necesito nada más, adiós", "finalizar", "hasta luego, buen día",
        "cierro la sesión",
    ],
    "fuera_de_alcance": [
        "¿cuánto cuesta una pizza?", "cuéntame un chiste", "¿qué clima hará mañana?",
        "¿quién ganó el partido de ayer?", "recomiéndame una película",
        "¿cuál es la capital de Francia?", "dame una receta de tacos", "¿cuánto es 15 por 4?",
        "quiero comprar boletos de avión", "¿dónde queda la farmacia más cercana?",
        "¿qué hora es en Tokio?", "háblame de la historia de México", "¿cómo me hago rico?",
        "¿cuál es el mejor restaurante de la ciudad?", "canta una canción",
        "¿quién es el presidente?", "quiero reservar un hotel", "¿cómo se baila salsa?",
        "¿cuánto cuesta un coche nuevo?", "dime un dato curioso", "¿qué signo soy?",
        "ayúdame con mi tarea de matemáticas", "¿cuándo es el próximo feriado?",
        "¿cómo preparo un café?",
    ],
}

### A2. Resumen del conjunto de datos

In [ ]:
filas = [(texto, intencion) for intencion, textos in DATOS.items() for texto in textos]
df = pd.DataFrame(filas, columns=["texto", "intencion"])
df.to_csv("datos_intenciones.csv", index=False, encoding="utf-8-sig")

print("Total de ejemplos:", len(df))
print("Intenciones:", df["intencion"].nunique())
display(df.sample(10, random_state=SEMILLA))

conteo = df["intencion"].value_counts()
display(conteo.to_frame("ejemplos").T)

conteo.plot(kind="bar", figsize=(9, 3), color="#1f3a5f")
plt.title("Ejemplos por intención")
plt.ylabel("Ejemplos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### A3. Preprocesamiento de texto

In [ ]:
stemmer = SnowballStemmer("spanish")
tokenizador = RegexpTokenizer(r"[a-z]+")

STOP = {
    "a", "al", "algo", "con", "como", "cual", "cuando", "de", "del", "el", "ella", "en",
    "es", "esta", "estan", "este", "esto", "la", "las", "le", "lo", "los", "me", "mi",
    "mis", "muy", "no", "o", "para", "pero", "por", "porque", "que", "se", "si", "sin",
    "su", "sus", "te", "tengo", "tiene", "un", "una", "uno", "y", "ya", "yo", "quiero",
    "necesito", "puedo", "estoy", "son", "hay", "mas", "tu", "tus", "nos", "ser", "soy",
}


def normalizar(texto):
    # Minúsculas y sin acentos
    texto = texto.lower().strip()
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")


def tokenizar(texto):
    return tokenizador.tokenize(texto)


def quitar_stopwords(tokens):
    return [t for t in tokens if t not in STOP]


def aplicar_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]


def preprocesar(texto):
    return aplicar_stemming(quitar_stopwords(tokenizar(normalizar(texto))))


ejemplo = "¿No puedo conectarme a la VPN desde mi casa?"
normalizado = normalizar(ejemplo)
tokens = tokenizar(normalizado)
sin_stop = quitar_stopwords(tokens)
raices = aplicar_stemming(sin_stop)

print("Texto original      :", ejemplo)
print("1. Normalización    :", normalizado)
print("2. Tokenización     :", tokens)
print("3. Sin stopwords    :", sin_stop)
print("4. Stemming         :", raices)

### A4. Representación del texto con TF-IDF

In [ ]:
X_ent_txt, X_pru_txt, y_ent, y_pru = train_test_split(
    df["texto"], df["intencion"], test_size=0.20, stratify=df["intencion"],
    random_state=SEMILLA)

vectorizador = TfidfVectorizer(tokenizer=preprocesar, lowercase=False,
                               token_pattern=None, ngram_range=(1, 2), sublinear_tf=True)
X_ent = vectorizador.fit_transform(X_ent_txt)
X_pru = vectorizador.transform(X_pru_txt)

print("Ejemplos de entrenamiento:", X_ent.shape[0])
print("Ejemplos de prueba       :", X_pru.shape[0])
print("Tamaño del vocabulario   :", X_ent.shape[1], "términos (raíces y pares de raíces)")

frase = "mi impresora no imprime"
vector = vectorizador.transform([frase])
terminos = vectorizador.get_feature_names_out()
pesos = sorted(zip(vector.indices, vector.data), key=lambda p: -p[1])
print()
print(f"Pesos TF-IDF de '{frase}':")
for indice, peso in pesos:
    print(f"  {terminos[indice]:<20} {peso:.3f}")

### A5. Entrenamiento y evaluación del clasificador de intenciones

In [ ]:
bayes = MultinomialNB(alpha=0.5).fit(X_ent, y_ent)
svm = LinearSVC(C=1.0, random_state=SEMILLA).fit(X_ent, y_ent)

exac_bayes = accuracy_score(y_pru, bayes.predict(X_pru))
exac_svm = accuracy_score(y_pru, svm.predict(X_pru))
print(f"Accuracy en prueba - Naive Bayes: {exac_bayes:.3f}")
print(f"Accuracy en prueba - SVM lineal : {exac_svm:.3f}")

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
modelo_cv = make_pipeline(
    TfidfVectorizer(tokenizer=preprocesar, lowercase=False, token_pattern=None,
                    ngram_range=(1, 2), sublinear_tf=True),
    LinearSVC(C=1.0, random_state=SEMILLA))
puntajes = cross_val_score(modelo_cv, df["texto"], df["intencion"], cv=validacion)
print(f"Validación cruzada (5 pliegues), SVM: {puntajes.mean():.3f} +/- {puntajes.std():.3f}")

# El chatbot usa el clasificador con mejor accuracy en prueba
clasificador = svm if exac_svm >= exac_bayes else bayes
print("Clasificador del chatbot:", type(clasificador).__name__)

pred = clasificador.predict(X_pru)
print()
print(classification_report(y_pru, pred, digits=3, zero_division=0))

In [ ]:
etiquetas = sorted(df["intencion"].unique())
matriz = confusion_matrix(y_pru, pred, labels=etiquetas)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matriz, display_labels=etiquetas).plot(
    ax=ax, cmap="Blues", xticks_rotation=60, colorbar=False)
ax.set_title("Matriz de confusión - intenciones (conjunto de prueba)")
plt.tight_layout()
plt.show()

errores = [(t, r, p) for t, r, p in zip(X_pru_txt, y_pru, pred) if r != p]
print(f"Consultas mal clasificadas: {len(errores)} de {len(y_pru)}")
for texto, real, predicha in errores[:10]:
    print(f"  '{texto}'  real={real}  predicha={predicha}")

## Parte B. Componente visual

El clasificador visual reconoce el tipo de equipo que aparece en una foto (impresora, router, laptop o monitor),
para relacionarlo con las intenciones de soporte del chatbot.

### B1. Obtención de las imágenes

In [ ]:
CLASES = {
    "impresora": ["laser printer", "inkjet printer", "office printer", "HP LaserJet"],
    "router": ["wireless router", "wifi router", "TP-Link router", "Linksys router"],
    "laptop": ["laptop computer", "notebook computer", "Dell laptop", "ThinkPad laptop"],
    "monitor": ["computer monitor", "LCD monitor", "flat panel computer display", "Dell monitor"],
}
OBJETIVO = 120          # imágenes por clase
CARPETA = "imagenes"
API = "https://commons.wikimedia.org/w/api.php"
CABECERA = {"User-Agent": "ProyectoEscolarChatbotTI/1.0 (trabajo academico)"}


def buscar_urls(consulta, maximo=80):
    # Busca archivos JPEG en Wikimedia Commons y regresa la URL de una miniatura
    urls = []
    parametros = {"action": "query", "format": "json", "generator": "search",
                  "gsrsearch": f"{consulta} filetype:bitmap", "gsrnamespace": 6,
                  "gsrlimit": 50, "prop": "imageinfo", "iiprop": "url|mime",
                  "iiurlwidth": 400}
    while len(urls) < maximo:
        respuesta = requests.get(API, params=parametros, headers=CABECERA, timeout=30)
        respuesta.raise_for_status()
        datos = respuesta.json()
        for pagina in datos.get("query", {}).get("pages", {}).values():
            info = pagina.get("imageinfo", [{}])[0]
            if info.get("mime") == "image/jpeg" and info.get("thumburl"):
                urls.append(info["thumburl"])
        if "continue" in datos:
            parametros.update(datos["continue"])
        else:
            break
    return urls


def descargar_clase(clase, consultas, objetivo):
    carpeta = os.path.join(CARPETA, clase)
    os.makedirs(carpeta, exist_ok=True)
    guardadas = len(os.listdir(carpeta))
    urls_vistas, huellas = set(), set()
    for consulta in consultas:
        for url in buscar_urls(consulta):
            if guardadas >= objetivo:
                return guardadas
            if url in urls_vistas:
                continue
            urls_vistas.add(url)
            try:
                contenido = requests.get(url, headers=CABECERA, timeout=30).content
                imagen = Image.open(io.BytesIO(contenido)).convert("RGB")
                huella = hashlib.md5(imagen.resize((32, 32)).tobytes()).hexdigest()
                if huella in huellas:
                    continue
                huellas.add(huella)
                imagen.thumbnail((400, 400))
                imagen.save(os.path.join(carpeta, f"{clase}_{guardadas:03d}.jpg"), quality=90)
                guardadas += 1
            except Exception:
                continue
            time.sleep(0.2)
    return guardadas


resumen = {}
for clase, consultas in CLASES.items():
    resumen[clase] = descargar_clase(clase, consultas, OBJETIVO)

print("Imágenes por clase:")
for clase, total in resumen.items():
    print(f"  {clase:<10} {total}")
print("Total:", sum(resumen.values()))

### B2. Muestra del conjunto de imágenes

In [ ]:
nombres = sorted(CLASES)
columnas = 6
fig, ejes = plt.subplots(len(nombres), columnas, figsize=(2.2 * columnas, 2.2 * len(nombres)))
for i, clase in enumerate(nombres):
    archivos = sorted(os.listdir(os.path.join(CARPETA, clase)))[:columnas]
    for j in range(columnas):
        ejes[i][j].axis("off")
        if j < len(archivos):
            ejes[i][j].imshow(Image.open(os.path.join(CARPETA, clase, archivos[j])))
    ejes[i][0].set_title(clase, loc="left", fontsize=11)
plt.tight_layout()
plt.show()

### B3. Depuración y preparación de los datos (entrenamiento, validación y prueba)

Si en la muestra anterior aparece una imagen que no corresponde a su clase, escribe su nombre de archivo
en la lista ELIMINAR (por ejemplo "router_012.jpg") y ejecuta la celda.

In [ ]:
ELIMINAR = []

for clase in CLASES:
    for archivo in ELIMINAR:
        ruta_archivo = os.path.join(CARPETA, clase, archivo)
        if os.path.exists(ruta_archivo):
            os.remove(ruta_archivo)
            print("Eliminada:", ruta_archivo)

In [ ]:
rutas, etiquetas_img = [], []
for indice, clase in enumerate(nombres):
    for archivo in sorted(os.listdir(os.path.join(CARPETA, clase))):
        rutas.append(os.path.join(CARPETA, clase, archivo))
        etiquetas_img.append(indice)

r_ent, r_tmp, e_ent, e_tmp = train_test_split(
    rutas, etiquetas_img, test_size=0.30, stratify=etiquetas_img, random_state=SEMILLA)
r_val, r_pru, e_val, e_pru = train_test_split(
    r_tmp, e_tmp, test_size=0.50, stratify=e_tmp, random_state=SEMILLA)

TAM = (224, 224)
LOTE = 32


def leer(ruta, etiqueta):
    imagen = tf.io.read_file(ruta)
    imagen = tf.io.decode_jpeg(imagen, channels=3)
    imagen = tf.image.resize(imagen, TAM)
    return keras.applications.resnet50.preprocess_input(imagen), etiqueta


def crear_dataset(rutas_, etiquetas_, mezclar=False):
    ds = tf.data.Dataset.from_tensor_slices((rutas_, np.array(etiquetas_)))
    if mezclar:
        ds = ds.shuffle(len(rutas_), seed=SEMILLA)
    return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(LOTE).prefetch(tf.data.AUTOTUNE)


ds_ent = crear_dataset(r_ent, e_ent, mezclar=True)
ds_val = crear_dataset(r_val, e_val)
ds_pru = crear_dataset(r_pru, e_pru)

print("Clases:", nombres)
print(f"Entrenamiento: {len(r_ent)} | Validación: {len(r_val)} | Prueba: {len(r_pru)}")

### B4. Arquitectura: ResNet50 con transferencia de aprendizaje

In [ ]:
base = keras.applications.ResNet50(weights="imagenet", include_top=False,
                                   input_shape=(224, 224, 3), pooling="avg")
base.trainable = False      # se conservan los pesos aprendidos en ImageNet

aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
], name="aumento_de_datos")

entrada = keras.Input(shape=(224, 224, 3))
x = aumento(entrada)
x = base(x, training=False)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(len(nombres), activation="softmax", name="clasificador")(x)

modelo = keras.Model(entrada, salida, name="resnet50_equipos_ti")
modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

### B5. Entrenamiento

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
historia = modelo.fit(ds_ent, validation_data=ds_val, epochs=15, callbacks=[parada],
                      shuffle=False, verbose=2)

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
ejes[0].plot(historia.history["loss"], label="entrenamiento")
ejes[0].plot(historia.history["val_loss"], label="validación")
ejes[0].set_title("Pérdida")
ejes[0].set_xlabel("Época")
ejes[0].legend()
ejes[1].plot(historia.history["accuracy"], label="entrenamiento")
ejes[1].plot(historia.history["val_accuracy"], label="validación")
ejes[1].set_title("Accuracy")
ejes[1].set_xlabel("Época")
ejes[1].legend()
plt.tight_layout()
plt.show()

### B6. Evaluación en el conjunto de prueba

In [ ]:
probabilidades = modelo.predict(ds_pru, verbose=0)
y_real = np.array(e_pru)
y_pred = probabilidades.argmax(axis=1)

exactitud = accuracy_score(y_real, y_pred)
p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    y_real, y_pred, average="macro", zero_division=0)

print(f"Accuracy : {exactitud:.3f}")
print(f"Precision (macro): {p_macro:.3f}")
print(f"Recall    (macro): {r_macro:.3f}")
print(f"F1-score  (macro): {f_macro:.3f}")
print()
print(classification_report(y_real, y_pred, target_names=nombres, digits=3, zero_division=0))

### B7. Matriz de confusión del clasificador visual

In [ ]:
matriz_img = confusion_matrix(y_real, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(matriz_img, display_labels=nombres).plot(
    ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión - imágenes (conjunto de prueba)")
plt.tight_layout()
plt.show()

## Parte C. Integración multimodal

El sistema recibe texto, una imagen o ambos. El texto pasa por el clasificador de intenciones y la imagen por la CNN;
la clase detectada en la imagen se relaciona con una intención de soporte para responder con el mismo catálogo de respuestas.

In [ ]:
RESPUESTAS = {
    "saludo": ["¡Hola! Soy el asistente de soporte de TI. ¿En qué te puedo ayudar?",
               "¡Buen día! Cuéntame tu problema técnico y te ayudo."],
    "ayuda": ["Puedo ayudarte con: contraseñas, VPN, internet/Wi-Fi, impresoras, instalación de "
              "software, correo, equipo lento, pantallas, horario de soporte o crear un ticket. "
              "También puedes enviarme la foto de tu equipo."],
    "contrasena": ["Para restablecer tu contraseña entra al portal de autoservicio, elige 'Olvidé mi "
                   "contraseña' y sigue los pasos de verificación. Si tu cuenta sigue bloqueada, "
                   "escribe 'ticket'."],
    "vpn": ["Para la VPN: 1) revisa que tengas internet, 2) cierra y abre el cliente VPN, "
            "3) verifica tu usuario y contraseña. Si el error continúa, escribe 'ticket'."],
    "wifi": ["Para problemas de red: apaga y enciende el Wi-Fi de tu equipo, olvida la red y vuelve a "
             "conectarte. Revisa también que el router tenga sus luces encendidas. Si otros equipos "
             "tampoco tienen internet, escribe 'ticket'."],
    "impresora": ["Revisa que la impresora esté encendida y con papel, y que sea la predeterminada en "
                  "tu equipo. Prueba cancelar la cola de impresión y volver a imprimir."],
    "software": ["La instalación de software se solicita por ticket, indicando el nombre del programa "
                 "y el motivo. Solo se instalan programas autorizados por TI."],
    "correo": ["Si tienes problemas con el correo, cierra y vuelve a abrir Outlook y revisa que tu "
               "buzón no esté lleno. Si no se soluciona, escribe 'ticket'."],
    "equipo_lento": ["Si tu equipo va lento: reinícialo, cierra programas que no uses y libera espacio "
                     "en disco. Si sigue igual, genera un ticket."],
    "pantalla": ["Revisa que el monitor esté encendido y que los cables de energía y de video (HDMI o "
                 "DisplayPort) estén bien conectados. Prueba con otro cable o puerto; si no hay "
                 "imagen, escribe 'ticket'."],
    "horario": ["El soporte de TI atiende de lunes a viernes de 8:00 a 18:00 h."],
    "ticket": ["Claro, describe brevemente el problema para generar tu ticket."],
    "agradecimiento": ["¡Con gusto! ¿Necesitas algo más?"],
    "despedida": ["¡Hasta luego! Que tengas un excelente día."],
}
SIN_ENTENDER = ("No logré entender tu consulta. Puedes escribir 'ayuda' para ver los temas que "
                "atiendo o enviarme una foto del equipo.")

# Relación entre la clase detectada en la imagen y la intención de soporte
CLASE_A_INTENCION = {"impresora": "impresora", "router": "wifi",
                     "laptop": "equipo_lento", "monitor": "pantalla"}
NOMBRE_EQUIPO = {"impresora": "una impresora", "router": "un router", "laptop": "una laptop",
                 "monitor": "un monitor"}
INTENCION_A_CLASE = {v: k for k, v in CLASE_A_INTENCION.items()}
UMBRAL_IMAGEN = 0.60
INTENCIONES_GENERALES = {"saludo", "ayuda", "agradecimiento", "despedida", "fuera_de_alcance"}


def clasificar_texto(texto):
    if not preprocesar(texto):
        return "fuera_de_alcance"
    return clasificador.predict(vectorizador.transform([texto]))[0]


def clasificar_imagen(ruta):
    imagen, _ = leer(ruta, 0)
    probas = modelo.predict(tf.expand_dims(imagen, 0), verbose=0)[0]
    indice = int(probas.argmax())
    return nombres[indice], float(probas[indice])


def responder_texto(texto, estado):
    if estado.get("esperando_ticket"):
        estado["esperando_ticket"] = False
        folio = f"TKT-{random.randint(1000, 9999)}"
        return "ticket_descripcion", (f"Listo, generé tu ticket {folio} con la descripción: "
                                      f"'{texto.strip()}'. TI te contactará pronto.")
    intencion = clasificar_texto(texto)
    if intencion == "fuera_de_alcance":
        return intencion, SIN_ENTENDER
    if intencion == "ticket":
        estado["esperando_ticket"] = True
    return intencion, random.choice(RESPUESTAS[intencion])


def responder(texto=None, ruta_imagen=None, estado=None):
    # Regresa un diccionario con la intención, la clase de imagen y la respuesta del chatbot
    estado = estado if estado is not None else {}
    resultado = {"intencion": None, "clase_imagen": None, "confianza": None}
    respuesta_texto = None

    if texto:
        resultado["intencion"], respuesta_texto = responder_texto(texto, estado)
    if ruta_imagen:
        clase, confianza = clasificar_imagen(ruta_imagen)
        resultado["clase_imagen"], resultado["confianza"] = clase, confianza

    if ruta_imagen and resultado["confianza"] < UMBRAL_IMAGEN:
        respuesta = ("No logré identificar el equipo en la foto con suficiente seguridad. "
                     "¿Puedes enviar otra imagen más cercana o describir el problema?")
    elif ruta_imagen and (not texto or resultado["intencion"] in INTENCIONES_GENERALES):
        intencion_img = CLASE_A_INTENCION[resultado["clase_imagen"]]
        respuesta = (f"Por la imagen, parece {NOMBRE_EQUIPO[resultado['clase_imagen']]} "
                     f"(confianza {resultado['confianza']:.0%}). "
                     + RESPUESTAS[intencion_img][0])
    elif ruta_imagen:
        equipo = NOMBRE_EQUIPO[resultado["clase_imagen"]]
        esperado = INTENCION_A_CLASE.get(resultado["intencion"])
        if esperado is None or esperado == resultado["clase_imagen"]:
            nota = f"La foto muestra {equipo} (confianza {resultado['confianza']:.0%})."
        else:
            nota = (f"La foto parece {equipo} ({resultado['confianza']:.0%}), que no coincide con "
                    "el tema de tu mensaje; dime si te refieres a ese equipo.")
        respuesta = f"{respuesta_texto} ({nota})"
    else:
        respuesta = respuesta_texto
    resultado["respuesta"] = respuesta
    return resultado

### C1. Pruebas del sistema integrado

In [ ]:
def elegir_prueba(clase):
    indice = nombres.index(clase)
    return next(r for r, e in zip(r_pru, e_pru) if e == indice)


def ejecutar(titulo, pasos):
    print("=" * 70)
    print(titulo)
    print("=" * 70)
    estado = {}
    for texto, ruta in pasos:
        if ruta:
            plt.figure(figsize=(2.6, 2.6))
            plt.imshow(Image.open(ruta))
            plt.axis("off")
            plt.show()
        if texto:
            print("Usuario (texto) :", texto)
        if ruta:
            print("Usuario (imagen):", os.path.basename(ruta))
        r = responder(texto, ruta, estado)
        if texto:
            print("  [intención detectada :", r["intencion"], "]")
        if ruta:
            print(f"  [clase en la imagen  : {r['clase_imagen']} ({r['confianza']:.0%})]")
        print("Chatbot:", r["respuesta"])
        print()


ejecutar("Prueba 1: texto y después imagen", [
    ("Mi impresora no imprime", None),
    (None, elegir_prueba("impresora")),
])

In [ ]:
ejecutar("Prueba 2: solo imagen (monitor)", [
    ("Hola", None),
    (None, elegir_prueba("monitor")),
])

ejecutar("Prueba 3: texto general con imagen (router)", [
    ("Necesito ayuda con este equipo", elegir_prueba("router")),
    ("Gracias", None),
])

ejecutar("Prueba 4: texto de soporte con imagen de laptop", [
    ("Se traba todo cuando abro varios programas", elegir_prueba("laptop")),
])